# Build `scaling_470m` once (CPU / High-RAM Colab)
This notebook uploads the project archive and frozen tokenizer, builds the 45/20/15/10/10 ToolCall base mix, and writes shards under local `/content`. It does not mount Google Drive. The first source check clones APIs-guru into transient local cache. Salesforce xLAM remains reserved for SFT. Do not select a GPU runtime.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, os, shutil, signal, subprocess, sys, zipfile
print('Upload ToolCall-Scaling-Colab-Complete.zip')
uploaded = files.upload()
archive = next(Path(name) for name in uploaded if name.endswith('.zip'))
workspace = Path('/content/toolcall_scaling_workspace')
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as zf: zf.extractall(workspace)
PROJECT = workspace / 'experiments/scaling/scaling_data'
assert PROJECT.is_dir(), PROJECT
%cd {PROJECT}

In [ ]:
ENV_ROOT = Path('/content/toolcall_scaling_data_env')
READY = ENV_ROOT / 'READY'
requirements_digest = hashlib.sha256(Path('requirements.txt').read_bytes()).hexdigest()
if ENV_ROOT.exists() and (not READY.is_file() or READY.read_text().strip() != requirements_digest):
    print('Removing incomplete or outdated data environment:', ENV_ROOT)
    shutil.rmtree(ENV_ROOT)
if not (ENV_ROOT / 'bin/python').is_file():
    free_gib = shutil.disk_usage('/content').free / 1024**3
    print(f'Creating pinned data environment; free disk: {free_gib:.1f} GiB')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--retries', '10', '--timeout', '60', 'virtualenv==20.29.3'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(ENV_ROOT)], check=True)
DATA_PYTHON = str(ENV_ROOT / 'bin/python')
subprocess.run([DATA_PYTHON, '-m', 'pip', '--version'], check=True)
if not READY.is_file():
    subprocess.run([DATA_PYTHON, '-m', 'pip', 'install', '--upgrade', '--retries', '10', '--timeout', '60', 'pip', 'setuptools', 'wheel'], check=True)
    subprocess.run([DATA_PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '--prefer-binary', '--retries', '10', '--timeout', '60', '-r', 'requirements.txt'], check=True)
    subprocess.run([DATA_PYTHON, 'scripts/validate_data_runtime.py'], check=True)
    READY.write_text(requirements_digest + '\n')
else:
    subprocess.run([DATA_PYTHON, 'scripts/validate_data_runtime.py'], check=True)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('Using HF_TOKEN from Colab secrets.')
else:
    print('No HF_TOKEN secret found. Public downloads still work, but with lower rate limits.')
print('Upload the frozen toolcall_spm_32k.model')
uploaded = files.upload()
model_name = next(name for name in uploaded if name.endswith('.model'))
target = PROJECT / 'artifacts/tokenizer/toolcall_spm_32k.model'
shutil.copy2(model_name, target)
print('Tokenizer:', target, target.stat().st_size, 'bytes')

## Checkpoint 1 — source and tokenizer smoke test
Stop if any source fails or if the 1M smoke build does not end with a `COMPLETE` marker. A strict verifier always runs before an output is accepted.

In [ ]:
def run_builder(arguments, output_root):
    result = subprocess.run([DATA_PYTHON, 'scripts/build_scaling_data.py', *arguments])
    complete = (output_root / 'COMPLETE').is_file()
    building = (output_root / 'BUILDING').is_file()
    print(f'Builder return code: {result.returncode}; output={output_root}; exists={output_root.exists()}; BUILDING={building}; COMPLETE={complete}')
    if result.returncode == -signal.SIGABRT and complete:
        print('Builder aborted only during interpreter shutdown; validating completed files now.')
    elif result.returncode != 0:
        result.check_returncode()
    if not complete:
        raise RuntimeError(f'Builder returned without a complete bundle at {output_root}')

SMOKE_ROOT = Path('/content/toolcall_scaling_data/_smoke_1m')
if SMOKE_ROOT.exists():
    print('Removing previous known smoke build:', SMOKE_ROOT)
    shutil.rmtree(SMOKE_ROOT)
subprocess.run([DATA_PYTHON, 'scripts/check_sources.py'], check=True)
run_builder(['--tokenizer', 'artifacts/tokenizer/toolcall_spm_32k.model', '--output', str(SMOKE_ROOT), '--limit-train-tokens', '1000000'], SMOKE_ROOT)
subprocess.run([DATA_PYTHON, 'scripts/verify_scaling_data.py', str(SMOKE_ROOT), '--checksums'], check=True)

## Full build
This writes about 952 MB of token binaries plus manifests to local Colab storage. If the destination already exists, inspect it first; the builder will not overwrite it silently. Transient Hugging Face HTTP disconnects are retried and resumed automatically.

In [ ]:
DATA_ROOT = Path('/content/toolcall_scaling_data/scaling_470m')
print('Output:', DATA_ROOT)
if (DATA_ROOT / 'COMPLETE').is_file():
    print('A completed bundle already exists; skipping generation and verifying it below.')
elif DATA_ROOT.exists():
    if (DATA_ROOT / 'BUILDING').is_file():
        print('Removing known incomplete build before restart:', DATA_ROOT)
        shutil.rmtree(DATA_ROOT)
    else:
        raise RuntimeError(f'Refusing to replace unrecognized directory: {DATA_ROOT}')
if not (DATA_ROOT / 'COMPLETE').is_file():
    run_builder(['--config', 'configs/scaling_470m.json', '--tokenizer', 'artifacts/tokenizer/toolcall_spm_32k.model', '--output', str(DATA_ROOT)], DATA_ROOT)

## Checkpoint 2 — final verification
The build is ready only if all three splits and the 32k tokenizer pass. Because `/content` is temporary, export the verified bundle before ending the runtime.

In [ ]:
subprocess.run([DATA_PYTHON, 'scripts/verify_scaling_data.py', str(DATA_ROOT)], check=True)
subprocess.run([DATA_PYTHON, 'scripts/inspect_scaling_data.py', str(DATA_ROOT), '--split', 'validation_general', '--samples', '3', '--tokens', '256'], check=True)
subprocess.run([DATA_PYTHON, 'scripts/inspect_scaling_data.py', str(DATA_ROOT), '--split', 'validation_structured', '--samples', '2', '--tokens', '256'], check=True)

## Export without Google Drive
Create one archive and download it before the Colab runtime ends. Keep the archive for the M13/M30/M60 training notebooks.

In [ ]:
EXPORT_ARCHIVE = Path('/content/scaling_470m.tar.gz')
if EXPORT_ARCHIVE.exists(): EXPORT_ARCHIVE.unlink()
subprocess.run(['tar', '-C', str(DATA_ROOT.parent), '-czf', str(EXPORT_ARCHIVE), DATA_ROOT.name], check=True)
print('Archive:', EXPORT_ARCHIVE, EXPORT_ARCHIVE.stat().st_size, 'bytes')
archive_sha = subprocess.check_output(['sha256sum', str(EXPORT_ARCHIVE)], text=True).split()[0]
print('SHA-256:', archive_sha)
files.download(str(EXPORT_ARCHIVE))